In [0]:
# %sql
# ALTER TABLE citibike.02_silver.citibikes
# SET TBLPROPERTIES (
#     'delta.enableChangeDataFeed' = 'true'
# );

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable


# --------------------------------------------------
# CONFIGURACIÓN
# --------------------------------------------------

SILVER_TABLE = "citibike.02_silver.citibikes"

DIM_TIME = "citibike.03_gold.dim_time"
DIM_STATION = "citibike.03_gold.dim_station"
DIM_BIKE = "citibike.03_gold.dim_bike_type"
DIM_MEMBER = "citibike.03_gold.dim_member_type"
FACT_TRIPS = "citibike.03_gold.fact_trips"

CHECKPOINT = (
    "/Volumes/citibike/01_bronze/checkpoints/silver_to_gold"
)


# --------------------------------------------------
# FUNCIONES AUXILIARES
# --------------------------------------------------

def has_rows(df):
    return len(df.take(1)) > 0


def merge_dimension(df, table_name, condition, update_columns):
    """
    Inserta nuevas filas y actualiza las existentes
    en una dimensión.
    """

    if not has_rows(df):
        return

    target = DeltaTable.forName(df.sparkSession, table_name)

    update_values = {
        column: f"source.{column}"
        for column in update_columns
    }

    insert_values = {
        column: f"source.{column}"
        for column in update_columns
    }

    (
        target.alias("target")
        .merge(
            df.alias("source"),
            condition
        )
        .whenMatchedUpdate(set=update_values)
        .whenNotMatchedInsert(values=insert_values)
        .execute()
    )


# --------------------------------------------------
# DIM_TIME
# --------------------------------------------------

def build_dim_time(df):

    months = {
        1: "enero",
        2: "febrero",
        3: "marzo",
        4: "abril",
        5: "mayo",
        6: "junio",
        7: "julio",
        8: "agosto",
        9: "septiembre",
        10: "octubre",
        11: "noviembre",
        12: "diciembre"
    }

    days = {
        1: "domingo",
        2: "lunes",
        3: "martes",
        4: "miércoles",
        5: "jueves",
        6: "viernes",
        7: "sábado"
    }

    month_expr = F.create_map(
        *[
            item
            for pair in months.items()
            for item in (F.lit(pair[0]), F.lit(pair[1]))
        ]
    )

    day_expr = F.create_map(
        *[
            item
            for pair in days.items()
            for item in (F.lit(pair[0]), F.lit(pair[1]))
        ]
    )

    result = (
        df
        .filter(F.col("started_at").isNotNull())
        .withColumn("start_datetime", F.col("started_at"))
        .withColumn("start_date", F.to_date("started_at"))
        .withColumn("year", F.year("started_at"))
        .withColumn("month", F.month("started_at"))
        .withColumn("day", F.dayofmonth("started_at"))
        .withColumn("hour", F.hour("started_at"))
        .withColumn("minute", F.minute("started_at"))
        .withColumn("day_of_week", F.dayofweek("started_at"))
        .withColumn(
            "day_name",
            F.element_at(day_expr, F.col("day_of_week"))
        )
        .withColumn(
            "month_name",
            F.element_at(month_expr, F.col("month"))
        )
        .withColumn(
            "time_id",
            F.date_format("started_at", "yyyyMMddHHmmss")
        )
        .select(
            "time_id",
            "start_datetime",
            "start_date",
            "year",
            "month",
            "day",
            "hour",
            "minute",
            "day_of_week",
            "day_name",
            "month_name"
        )
        .dropDuplicates(["time_id"])
    )

    return result


# --------------------------------------------------
# DIM_STATION
# --------------------------------------------------

def build_dim_station(df):

    start_stations = df.select(
        F.col("start_station_id").cast("string").alias("station_id"),
        F.col("start_station_name").alias("station_name"),
        F.col("start_lat").cast("double").alias("latitude"),
        F.col("start_lng").cast("double").alias("longitude"),
        F.col("_commit_version"),
        F.col("ingestion_at")
    )

    end_stations = df.select(
        F.col("end_station_id").cast("string").alias("station_id"),
        F.col("end_station_name").alias("station_name"),
        F.col("end_lat").cast("double").alias("latitude"),
        F.col("end_lng").cast("double").alias("longitude"),
        F.col("_commit_version"),
        F.col("ingestion_at")
    )

    stations = (
        start_stations
        .unionByName(end_stations)
        .filter(F.col("station_id").isNotNull())
    )

    # Si una estación aparece varias veces, conservar
    # la información de la versión más reciente.
    window = (
        Window
        .partitionBy("station_id")
        .orderBy(
            F.col("_commit_version").desc(),
            F.col("ingestion_at").desc_nulls_last(),
            F.col("station_name").desc_nulls_last()
        )
    )

    stations = (
        stations
        .withColumn("row_num", F.row_number().over(window))
        .filter(F.col("row_num") == 1)
        .withColumn("station_key", F.col("station_id"))
        .select(
            "station_key",
            "station_id",
            "station_name",
            "latitude",
            "longitude"
        )
    )

    return stations


# --------------------------------------------------
# DIM_BIKE_TYPE
# --------------------------------------------------

def build_dim_bike_type(df):

    return (
        df
        .filter(F.col("rideable_type").isNotNull())
        .select(
            F.col("rideable_type").cast("string").alias("bike_type_key"),
            F.col("rideable_type").cast("string").alias("rideable_type")
        )
        .dropDuplicates(["bike_type_key"])
    )


# --------------------------------------------------
# DIM_MEMBER_TYPE
# --------------------------------------------------

def build_dim_member_type(df):

    return (
        df
        .filter(F.col("member_casual").isNotNull())
        .select(
            F.col("member_casual").cast("string").alias("member_type_key"),
            F.col("member_casual").cast("string").alias("member_casual")
        )
        .dropDuplicates(["member_type_key"])
    )


# --------------------------------------------------
# FACT_TRIPS
# --------------------------------------------------

def build_fact_trips(df):

    return (
        df
        .filter(
            F.col("ride_id").isNotNull()
            & F.col("started_at").isNotNull()
        )
        .withColumn(
            "time_id",
            F.date_format("started_at", "yyyyMMddHHmmss")
        )
        .withColumn(
            "bike_type_key",
            F.col("rideable_type").cast("string")
        )
        .withColumn(
            "member_type_key",
            F.col("member_casual").cast("string")
        )
        .withColumn(
            "start_station_key",
            F.col("start_station_id").cast("string")
        )
        .withColumn(
            "end_station_key",
            F.col("end_station_id").cast("string")
        )
        .withColumn(
            "trip_duration_minutes",
            (
                F.col("ended_at").cast("long")
                - F.col("started_at").cast("long")
            ) / 60.0
        )
        .filter(
            F.col("trip_duration_minutes").isNull()
            | (F.col("trip_duration_minutes") >= 0)
        )
        .select(
            "ride_id",
            "time_id",
            "bike_type_key",
            "member_type_key",
            "start_station_key",
            "end_station_key",
            "started_at",
            "ended_at",
            "trip_duration_minutes"
        )
    )


# --------------------------------------------------
# PROCESAMIENTO DE CADA MICRO-BATCH
# --------------------------------------------------

def process_gold(microBatchDF, batch_id):

    spark = microBatchDF.sparkSession

    print(f"Procesando micro-batch Gold: {batch_id}")

    # Nos quedamos con los cambios relevantes.
    changes = microBatchDF.filter(
        F.col("_change_type").isin(
            "insert",
            "update_postimage",
            "delete"
        )
    )

    if not has_rows(changes):
        print("No hay cambios que procesar.")
        return

    # Para cada ride_id conservamos el último cambio.
    window = (
        Window
        .partitionBy("ride_id")
        .orderBy(F.col("_commit_version").desc())
    )

    latest_changes = (
        changes
        .withColumn("row_num", F.row_number().over(window))
        .filter(F.col("row_num") == 1)
        .drop("row_num")
    )

    # Separar altas/actualizaciones y eliminaciones.
    upserts = latest_changes.filter(
        F.col("_change_type").isin(
            "insert",
            "update_postimage"
        )
    )

    deletes = latest_changes.filter(
        F.col("_change_type") == "delete"
    )

    # --------------------------------------------------
    # 1. ACTUALIZAR DIMENSIONES
    # --------------------------------------------------

    if has_rows(upserts):

        # DIM_TIME
        dim_time_df = build_dim_time(upserts)

        merge_dimension(
            dim_time_df,
            DIM_TIME,
            "target.time_id = source.time_id",
            [
                "time_id",
                "start_datetime",
                "start_date",
                "year",
                "month",
                "day",
                "hour",
                "minute",
                "day_of_week",
                "day_name",
                "month_name"
            ]
        )

        # DIM_STATION
        dim_station_df = build_dim_station(upserts)

        merge_dimension(
            dim_station_df,
            DIM_STATION,
            "target.station_key = source.station_key",
            [
                "station_key",
                "station_id",
                "station_name",
                "latitude",
                "longitude"
            ]
        )

        # DIM_BIKE_TYPE
        dim_bike_df = build_dim_bike_type(upserts)

        merge_dimension(
            dim_bike_df,
            DIM_BIKE,
            "target.bike_type_key = source.bike_type_key",
            [
                "bike_type_key",
                "rideable_type"
            ]
        )

        # DIM_MEMBER_TYPE
        dim_member_df = build_dim_member_type(upserts)

        merge_dimension(
            dim_member_df,
            DIM_MEMBER,
            "target.member_type_key = source.member_type_key",
            [
                "member_type_key",
                "member_casual"
            ]
        )

        # --------------------------------------------------
        # 2. ACTUALIZAR FACT_TRIPS
        # --------------------------------------------------

        fact_df = build_fact_trips(upserts)

        if has_rows(fact_df):

            fact_table = DeltaTable.forName(
                spark,
                FACT_TRIPS
            )

            (
                fact_table.alias("target")
                .merge(
                    fact_df.alias("source"),
                    "target.ride_id = source.ride_id"
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )

    # --------------------------------------------------
    # 3. PROCESAR ELIMINACIONES
    # --------------------------------------------------

    if has_rows(deletes):

        delete_keys = (
            deletes
            .select("ride_id")
            .filter(F.col("ride_id").isNotNull())
            .dropDuplicates(["ride_id"])
        )

        fact_table = DeltaTable.forName(
            spark,
            FACT_TRIPS
        )

        (
            fact_table.alias("target")
            .merge(
                delete_keys.alias("source"),
                "target.ride_id = source.ride_id"
            )
            .whenMatchedDelete()
            .execute()
        )

    print(f"Micro-batch Gold {batch_id} procesado correctamente.")